# Quran Forced Align — All Reciters × 114 Surahs (Colab GPU)\n\n**Performance config (verified):** `--device cuda --cuda-batch-size 8 --intra-surah-split --max-workers 1`\n**Output:** Opus audio (+ ffmpeg normalize/compress) + word-level JSON per surah → Google Drive\n\n*Source reciter list / download URLs must be configured in Cell 4.*

In [ ]:
# 1. Install package + CUDA extra\n!pip install -q git+https://github.com/HsnSaboor/quran-forced-align.git#main --extra cuda 2>/dev/null || \
!pip install -q -e . --extra cuda
# Ensure model file present\nimport os, urllib.request\nos.makedirs('model', exist_ok=True)\nif not os.path.exists('model/zipformer_p_arabic_v2.int8.onnx'):\n    print('Download model (~73MB) manually: see repo model/ or HuggingFace Muno459/zipformer_p-arabic-v2')

In [ ]:
# 2. Mount Drive + setup dirs\nfrom google.colab import drive\ndrive.mount('/content/drive')\n\nDRIVE_ROOT = '/content/drive/MyDrive/quran_forced_align'\nAUDIO_DIR  = f'{DRIVE_ROOT}/audio_input'\nOUTPUT_DIR = f'{DRIVE_ROOT}/output'\nOPUS_DIR   = f'{OUTPUT_DIR}/opus'\nJSON_DIR   = f'{OUTPUT_DIR}/json'\n!mkdir -p {AUDIO_DIR} {OUTPUT_DIR} {OPUS_DIR} {JSON_DIR}

# 4. Reciter config + assabile download logic (from session-2026-08-14 network capture)
# Pattern discovered: /ajax/loadplayer-{rec_id}-{surah_id} -> JSON; /ajax/getrcita-link-{id} -> .mp3
# MP3 URL: media.assabile.com/assabile/recitations_{set}/mp3/{slug}-{surah:03d}-{name}-{coll_id}-{rec_id}.mp3
# Only process reciters with all 114 .mp3 files present. Pick primary set (collection 9 / murattal, data-default=1).

RECITERS = [
    {
        'name': 'abdul-rahman-al-sudais-12',
        'slug': 'abdul-rahman-al-sudais',
        'audio_dir': f'{AUDIO_DIR}/sudais',
        'reciter_id': 12,
        'enhance': True,
    },
    # Add full 114-surah reciters from assabile here after verifying 114 mp3s exist
]

SURAHS = list(range(1, 115))

def filter_full_reciters(reciter_list, base_audio_dir):
    """Only keep reciters with 114 .mp3 files (all surahs) present."""
    full = []
    for r in reciter_list:
        d = r.get('audio_dir', base_audio_dir + '/' + r['name'])
        files = [f for f in __import__('os').listdir(d) if f.endswith('.mp3')] if __import__('os').path.exists(d) else []
        if len(files) == 114:
            full.append(r)
            print(f'[FULL] {r["name"]}: 114 surahs OK')
        else:
            print(f'[SKIP] {r["name"]}: only {len(files)}/114 files — missing audio, skip')
    return full

# Filter before running alignment
RECITERS = filter_full_reciters(RECITERS, AUDIO_DIR)


In [ ]:
# 4. Reciter config\nRECITERS = [\n    {\n        'name': 'alafasy',\n        'audio_dir': f'{AUDIO_DIR}/alafasy',  # expects 001.mp3 ... 114.mp3\n        'enhance': True,\n    },\n    # Add more reciters from assabile.com here\n]\n\n# Example: download helper (run per reciter manually or batch)\ndef download_reciter_audio(reciter_name, surah_numbers, out_dir):\n    '''Placeholder — replace with real assabile.com URL logic or scp/wget.'''\n    import os\n    os.makedirs(out_dir, exist_ok=True)\n    for s in surah_numbers:\n        src = f'https://example.com/reciter/{reciter_name}/{s:03d}.mp3'\n        dst = f'{out_dir}/{s:03d}.mp3'\n        # !wget -q -O {dst} {src}  # uncomment with real URL\n    return out_dir\n\nSURAHS = list(range(1, 115))  # all 114 surahs

## 5. Audio Enhancement → Opus\n*Enhancement = ffmpeg loudnorm/compression + opus encode. This runs independently of alignment.*

In [ ]:
# 5a. Convert each reciter audio to Opus with optional normalize/compress\n!apt-get install -y -qq ffmpeg > /dev/null 2>&1\n\nimport subprocess, glob, os\n\ndef enhance_to_opus(mp3_path, opus_path, loudnorm=True):\n    os.makedirs(os.path.dirname(opus_path) or '.', exist_ok=True)\n    filters = 'loudnorm=I=-16:TP=-1.5:LRA=11,acompressor=threshold=-25dB:ratio=3:attack=5:release=50' if loudnorm else 'anull'\n    cmd = [\n        'ffmpeg', '-y', '-i', mp3_path,\n        '-af', filters,\n        '-c:a', 'libopus', '-b:a', '96k', '-vbr', 'on',\n        '-application', 'audio', '-frame_duration', '60',\n        opus_path\n    ]\n    subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)\n    return opus_path\n\n# Example per reciter batch\nfor r in RECITERS:\n    rec_opus = f'{OPUS_DIR}/{r["name"]}'\n    os.makedirs(rec_opus, exist_ok=True)\n    for s in SURAHS:\n        src_mp3 = f'{r["audio_dir"]}/{s:03d}.mp3'\n        dst_opus = f'{rec_opus}/{s:03d}.opus'\n        if os.path.exists(src_mp3) and (r.get('enhance') or not os.path.exists(dst_opus)):\n            enhance_to_opus(src_mp3, dst_opus, loudnorm=True)\n            print(f'[opus] {r["name"]} surah {s}')

## 6. Batch Forced Alignment (Verified GPU Config)\n**Settings:** `--device cuda --cuda-batch-size 8 --intra-surah-split --max-workers 1`\nThis is the empirically-verified single-GPU optimum (3.74x speedup vs serial, byte-identical output).

In [ ]:
# 6a. Run alignment per reciter (batch all 114 surahs)\nimport time\n\nBATCH_SIZE = 8           # increase if VRAM allows (Colab T4 ~8-12)\nMAX_WORKERS = 1           # NEVER >1 on single GPU (verified slower)\nDEVICE = 'cuda'\n\nfor r in RECITERS:\n    rec_name = r['name']\n    audio_dir = r['audio_dir']\n    out_dir = f'{JSON_DIR}/{rec_name}'\n    if not os.path.exists(audio_dir):\n        print(f'[SKIP] {rec_name}: audio_dir {audio_dir} missing — download first.')\n        continue\n    # Count how many .mp3 exist for this reciter\n    available = sorted([int(f.split('.')[0]) for f in os.listdir(audio_dir) if f.endswith('.mp3')])\n    if not available:\n        print(f'[SKIP] {rec_name}: no .mp3 files in {audio_dir}')\n        continue\n    print(f'[ALIGN] {rec_name}: {len(available)} surahs available (batch_size={BATCH_SIZE})')\n    # Build surah spec string\n    spec = '-'.join([f'{min(available)}-{max(available)}'])  # simple range; adapt if gaps\n    cmd = [\n        'python', '-m', 'quran_forced_align.batch_cli',\n        '--surahs', f'{min(available)}-{max(available)}',\n        '--audio-dir', audio_dir,\n        '--out-dir', out_dir,\n        '--device', DEVICE,\n        '--cuda-batch-size', str(BATCH_SIZE),\n        '--intra-surah-split',\n        '--max-workers', str(MAX_WORKERS),\n        '--model', 'model/zipformer_p_arabic_v2.int8.onnx',\n        '--tokens', 'model/tokens.txt',\n    ]\n    print(' '.join(cmd))\n    t0 = time.time()\n    ret = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)\n    elapsed = time.time() - t0\n    print(ret.stdout.decode('utf-8', errors='ignore')[-2000:] if ret.stdout else '')\n    print(f'[FINISH] {rec_name}: {elapsed:.1f}s  return_code={ret.returncode}')

## 7. Copy Outputs to Drive (structured)\nOrganize by reciter → opus audio + json word-timings.

In [ ]:
# 7a. Organize Drive output\nimport shutil, glob\n\nfor r in RECITERS:\n    rec = r['name']\n    src_opus = f'{OPUS_DIR}/{rec}'\n    dst_opus = f'{DRIVE_ROOT}/opus/{rec}'\n    src_json = f'{JSON_DIR}/{rec}'\n    dst_json = f'{DRIVE_ROOT}/json/{rec}'\n    os.makedirs(dst_opus, exist_ok=True)\n    os.makedirs(dst_json, exist_ok=True)\n    # Copy opus\n    if os.path.exists(src_opus):\n        for f in glob.glob(f'{src_opus}/*.opus'):\n            shutil.copy2(f, dst_opus)\n    # Copy JSON (already written by batch_cli)\n    if os.path.exists(src_json):\n        for f in glob.glob(f'{src_json}/*.json'):\n            shutil.copy2(f, dst_json)\n    print(f'[DRIVE] {rec}: opus={len(glob.glob(f"{dst_opus}/*.opus"))}  json={len(glob.glob(f"{dst_json}/*.json"))}')

## 8. Summary Report\nPrint per-reciter counts of opus + json files saved in Drive.

In [ ]:
# 8. Report\nprint('=== DRIVE OUTPUT SUMMARY ===')\nfor r in RECITERS:\n    rec = r['name']\n    opus_files = sorted(glob.glob(f'{DRIVE_ROOT}/opus/{rec}/*.opus'))\n    json_files = sorted(glob.glob(f'{DRIVE_ROOT}/json/{rec}/*.json'))\n    print(f'{rec:15s}  opus={len(opus_files):>3}  json={len(json_files):>3}  (expected 114 each)')

---\n### Notes\n- Replace download URLs in Cell 4 if pulling from assabile.com.\n- `--max-workers 1` is required for single Colab GPU; `--max-workers > 1` slows down due to CUDA context contention.\n- `--intra-surah-split` + `--cuda-batch-size 8` gives ~3.7x speedup with zero decode differences.\n- Audio enhancement (`ffmpeg` loudnorm + libopus) is independent of alignment; disable by setting `enhance=False`.